In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd

from libpysal.weights import Queen, Rook, KNN
import libpysal as lps
import spreg

import statsmodels.formula.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

In [ ]:
buildings = gpd.read_file('data/brooklyn_buildings_with_raster_values.shp')
buildings = buildings.to_crs('EPSG:26918')
df = pd.read_csv('data/merged.csv')
df = df.merge(buildings, on='uID')

In [ ]:
correlations = df.corr().unstack().sort_values(ascending=False)
correlations = pd.DataFrame(correlations).reset_index() 
correlations.columns = ['Variable1', 'Variable2', 'Correlation']
correlations = correlations.query("Variable1 == 'lst' & Variable2 != 'lst'").sort_values(by=['Correlation'], ascending=False, key=abs)
with pd.option_context('display.max_rows', None,):
    print(correlations)

In [ ]:
top_correlations = correlations[abs(correlations.Correlation) >= 0.10]
top_correlations

In [ ]:
df_select = df[['lst', 'ndvi', 'buildingBuildingAdjacency', 'albedo', 'buildingPerimeterWall', 'closeness', 'buildingVolumeFacadeRatio', 'streetOpenness', 'buildingSquareness', 'tessCircularCompactness', 'buildingSharedWallsRatio', 'geometry']]
df_select = df_select.replace(np.inf, np.nan)
df_select = df_select.dropna()

In [ ]:
def get_vif(df):
    vif_data = pd.DataFrame()
    vif_data["feature"] = df.columns[1:-1]

    vif_data['VIF'] = [variance_inflation_factor(df.iloc[:, 1:-1].values, i) for i in range(len(df.columns[1:-1]))]

    with pd.option_context('display.max_rows', None,):
        print(vif_data)
        
get_vif(df_select)

In [ ]:
w = Queen.from_dataframe(gpd.GeoDataFrame(df_select))

In [ ]:
y = np.array(df_select['lst'])
X = np.array(df_select.iloc[:, 1:-1])

In [ ]:
ols = spreg.OLS(y, X, w, name_y='LST', name_x=df_select.columns[1:-1].tolist(), name_ds='Brooklyn', nonspat_diag=True, spat_diag=True, moran=True, white_test=True)

In [ ]:
print(ols.summary)